# Kaggle League — Submission Notebook

| | |
|---|---|
| **Team name** | *Surname1 & Surname2* |
| **Edition** | *1 / 2 / 3 / 4* |
| **Dataset** | *classification / regression* |
| **Public score of the selected submission** | *fill in before uploading* |

> **This header cell is required by the Kaggle League rules.** Fill in all four fields before uploading the notebook to the course platform. Upload one notebook per dataset (two per edition).

## How to Use This Template

This notebook is a **starting point** that already satisfies every formal requirement of the Kaggle League (see the rules in the lecture slides):

- a header cell with team, edition, dataset and the public score;
- the required packages are listed in the first code cell;
- it loads the **raw** Kaggle files, does all preprocessing and feature engineering, and trains **only models of the edition's family** (an assertion enforces it);
- all random seeds are fixed, so the run is reproducible;
- it produces **exactly** the submission file selected for the private leaderboard, in the format of `sample_submission.csv`;
- it runs top-to-bottom without errors in a clean environment (`uv sync`).

What it does *not* do is win: the preprocessing is minimal, the features are the raw ones, and the hyperparameters are defaults. Everything between *Load data* and *Write the submission* is yours to improve — that is the competition. Keep the structure, replace the contents.

**Workflow for every edition:**

1. Copy this notebook, e.g. `kl_edition1_classification.ipynb`.
2. Download `train.csv`, `test.csv` and `sample_submission.csv` from the Kaggle competition page and set `DATA_DIR` below.
3. Set `EDITION`, `TASK` and `TEAM_NAME`.
4. Develop: features, validation, tuning. Compare every idea with the cross-validated score, **not** with the public leaderboard (30% of the test set — a noisy hint, and you have only 5 submissions per day).
5. Fit the final pipeline on all training data, write the submission, upload it to Kaggle.
6. Before the deadline: select your submission on Kaggle, paste its public score into the header, **restart the kernel and run all cells**, check that the regenerated file is identical to the one you selected (Section 7), upload the notebook.

Until the real competition files are available, the template runs on a small synthetic dataset shipped with the repository (`data/kaggle_demo/`), which has the same structure as a Kaggle competition.

In [ ]:
# Required packages (all part of the course environment: `uv sync`)
# numpy, pandas, scikit-learn, matplotlib
import hashlib
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

print(f"pandas {pd.__version__}, numpy {np.__version__}, scikit-learn {sklearn.__version__}")

# 1. Configuration

Everything that changes between editions and datasets lives in this one cell.

In [ ]:
# --- Team and edition ---------------------------------------------------------------------------
TEAM_NAME = "Surname1 & Surname2"
EDITION = 1  # 1: linear/logistic, 2: KNN, 3: SVM, 4: trees & random forest
TASK = "classification"  # "classification" (metric: ROC AUC) or "regression" (metric: RMSE)

# --- Files --------------------------------------------------------------------------------------
# Point this at the folder with the raw Kaggle files: train.csv, test.csv, sample_submission.csv
DATA_DIR = Path("..") / "data" / "kaggle_demo" / TASK
OUTPUT_DIR = Path(".") / "submissions"

# --- Reproducibility ----------------------------------------------------------------------------
SEED = 42
np.random.seed(SEED)

# --- Validation ---------------------------------------------------------------------------------
N_FOLDS = 5

assert TASK in ("classification", "regression")
assert EDITION in (1, 2, 3, 4)
print(f"Edition {EDITION} | {TASK} | data: {DATA_DIR.resolve()}")

# 2. Load the Raw Kaggle Files

The submission format is defined by `sample_submission.csv`: its first column is the identifier, its second column is the prediction. The training target has the same name as the prediction column.

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

ID_COLUMN, PREDICTION_COLUMN = sample_submission.columns[:2]
TARGET_COLUMN = PREDICTION_COLUMN

print(f"train: {train.shape}, test: {test.shape}, sample_submission: {sample_submission.shape}")
print(f"id column: '{ID_COLUMN}', target column: '{TARGET_COLUMN}'")

# Sanity checks on the files
assert TARGET_COLUMN in train.columns, "target column not found in train.csv"
assert TARGET_COLUMN not in test.columns, "test.csv must not contain the target"
assert set(test[ID_COLUMN]) == set(sample_submission[ID_COLUMN]), "test ids and sample_submission ids differ"
assert train[ID_COLUMN].is_unique and test[ID_COLUMN].is_unique, "ids must be unique"

X_train = train.drop(columns=[ID_COLUMN, TARGET_COLUMN])
y_train = train[TARGET_COLUMN]
X_test = test.drop(columns=[ID_COLUMN])

if TASK == "classification":
    assert set(y_train.unique()) == {0, 1}, "classification target must be 0/1"
    print(f"Share of positives: {y_train.mean():.4f}")
else:
    print(
        f"Target: mean {y_train.mean():.4f}, std {y_train.std():.4f}, min {y_train.min():.4f}, max {y_train.max():.4f}"
    )

train.head()

# 3. Quick Look at the Data

Keep this short in the submission notebook — put your real EDA in a separate notebook (see `eda_and_data_wrangling.ipynb` for the full workflow). What must be known before modelling: column types, missing values, cardinality of the categorical columns, and whether train and test look alike.

In [ ]:
numerical_columns = X_train.select_dtypes(include="number").columns.tolist()
categorical_columns = X_train.select_dtypes(exclude="number").columns.tolist()

overview = pd.DataFrame(
    {
        "dtype": X_train.dtypes.astype(str),
        "missing (train)": X_train.isna().mean().round(4),
        "missing (test)": X_test.isna().mean().round(4),
        "n_unique (train)": X_train.nunique(),
    }
)
print(overview)
print(f"\nNumerical features ({len(numerical_columns)}): {numerical_columns}")
print(f"Categorical features ({len(categorical_columns)}): {categorical_columns}")

# Categories present in test but not in train are a classic source of errors at prediction time
for column in categorical_columns:
    unseen = set(X_test[column].dropna()) - set(X_train[column].dropna())
    if unseen:
        print(f"  '{column}': categories only in test: {unseen}")

# 4. Preprocessing and Feature Engineering

Two rules from the labs:

1. **Stateless** feature engineering (ratios, logs, flags) goes into a function that is applied to train and test alike.
2. **Stateful** steps (imputation, encoding, scaling) go into a `ColumnTransformer` inside the model `Pipeline`, so that they are fitted inside every cross-validation fold and never see the validation part — no leakage, and the same object is applied to the test set.

Any preprocessing, feature engineering and feature selection is allowed in every edition. This is where most of the points are won.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def add_features(data):
    """Stateless feature engineering - applied identically to train and test. Extend freely."""
    data = data.copy()
    # Example: a log transformation of a right-skewed, non-negative column (remove if not applicable)
    for column in data.select_dtypes(include="number").columns:
        if data[column].min() >= 0 and data[column].skew() > 2:
            data[f"{column}_log1p"] = np.log1p(data[column])
    return data


X_train_fe = add_features(X_train)
X_test_fe = add_features(X_test)

numerical_columns = X_train_fe.select_dtypes(include="number").columns.tolist()
categorical_columns = X_train_fe.select_dtypes(exclude="number").columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]),
            numerical_columns,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                    ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=0.01)),
                ]
            ),
            categorical_columns,
        ),
    ],
    verbose_feature_names_out=False,
)

print(
    f"Features after engineering: {X_train_fe.shape[1]} ({len(numerical_columns)} numerical, {len(categorical_columns)} categorical)"
)

# 5. The Model — Restricted to the Edition's Family

The dictionary below mirrors the *allowed models* table of the rules. The assertion makes it impossible to submit predictions from a model outside the family by accident. Ensembles of models from the same family (e.g. averaging several KNN models) are allowed; anything else that *produces the submitted predictions* is not.

Replace the default model with your tuned one — but keep the assertion.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import (
    ElasticNet,
    Lasso,
    LinearRegression,
    LogisticRegression,
    Ridge,
    SGDClassifier,
    SGDRegressor,
)
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.svm import SVC, SVR, LinearSVC, LinearSVR, NuSVC, NuSVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

ALLOWED_MODELS = {
    1: (LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression, SGDRegressor, SGDClassifier),
    2: (KNeighborsClassifier, KNeighborsRegressor),
    3: (SVC, NuSVC, LinearSVC, SVR, NuSVR, LinearSVR),
    4: (
        DecisionTreeClassifier,
        DecisionTreeRegressor,
        RandomForestClassifier,
        RandomForestRegressor,
        ExtraTreesClassifier,
        ExtraTreesRegressor,
    ),
}

# Library defaults of the edition's family - the starting point. Tune them with cross-validation!
DEFAULT_MODELS = {
    (1, "classification"): LogisticRegression(max_iter=2000),
    (1, "regression"): Ridge(),
    (2, "classification"): KNeighborsClassifier(),
    (2, "regression"): KNeighborsRegressor(),
    # SVC has no predict_proba: Platt scaling via CalibratedClassifierCV gives the probabilities needed for ROC AUC
    (3, "classification"): CalibratedClassifierCV(SVC(random_state=SEED), ensemble=False),
    (3, "regression"): SVR(),
    (4, "classification"): RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
    (4, "regression"): RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1),
}

model = DEFAULT_MODELS[(EDITION, TASK)]

# The rule check: the estimator that produces the predictions must belong to the edition's family
# (a calibration wrapper is allowed - the predictions still come from the wrapped model)
core_model = model.estimator if isinstance(model, CalibratedClassifierCV) else model
assert isinstance(core_model, ALLOWED_MODELS[EDITION]), (
    f"{type(core_model).__name__} is not allowed in edition {EDITION}"
)

pipeline = Pipeline([("preprocessor", preprocessor), ("model", model)])
pipeline

# 6. Validation

The metric of the competition is **ROC AUC** (classification — submit *probabilities* of class 1, never hard labels) or **RMSE** (regression). Cross-validation on the training set is your leaderboard during development:

- it uses all the training data,
- it is far less noisy than the public leaderboard, and it does not consume submissions,
- the standard deviation across folds tells you whether a difference between two ideas is real.

We also compute the score of the **naive baseline** (a constant prediction) — the first bar to clear in the scoring table. Beating it is worth 2 points, beating the lecturer's default model (Benchmark I) 4 points, the tuned one (Benchmark II) 5.

In [ ]:
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score

if TASK == "classification":
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    scoring = "roc_auc"
    baseline = DummyClassifier(strategy="prior")
else:
    cv = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    scoring = "neg_root_mean_squared_error"
    baseline = DummyRegressor(strategy="mean")


def report(name, scores):
    sign = 1 if TASK == "classification" else -1
    print(
        f"{name:<42s} {scoring:>28s}: {sign * scores.mean():.4f} (± {scores.std():.4f})  folds: {np.round(sign * scores, 4)}"
    )


with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # the constant classifier produces harmless 'only one class' warnings
    baseline_scores = cross_val_score(baseline, X_train_fe, y_train, cv=cv, scoring=scoring)
model_scores = cross_val_score(pipeline, X_train_fe, y_train, cv=cv, scoring=scoring, n_jobs=-1)

print("=" * 100)
print(f"{N_FOLDS}-FOLD CROSS-VALIDATION ON THE TRAINING SET")
print("=" * 100)
report("Naive baseline (constant)", baseline_scores)
report(f"{type(model).__name__} (this notebook)", model_scores)

## Hyperparameter Tuning (Your Work Starts Here)

The cell below is a minimal example of a cross-validated search with the pipeline. Extend the grid, switch to `RandomizedSearchCV`, add feature selection steps — see the lab notebook of the edition's model family. Remember: `best_score_` is optimistically biased; the honest comparison between *different* ideas is the CV score of each final pipeline.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Example grids per edition - deliberately tiny so that the template runs in seconds. Replace them.
EXAMPLE_GRIDS = {
    1: {"model__C": [0.1, 1.0, 10.0]} if TASK == "classification" else {"model__alpha": [0.1, 1.0, 10.0]},
    2: {"model__n_neighbors": [5, 15, 31], "model__weights": ["uniform", "distance"]},
    3: {"model__estimator__C": [0.3, 1.0, 3.0]} if TASK == "classification" else {"model__C": [0.3, 1.0, 3.0]},
    4: {"model__max_features": ["sqrt", 0.5], "model__min_samples_leaf": [1, 5]},
}

search = GridSearchCV(pipeline, EXAMPLE_GRIDS[EDITION], cv=cv, scoring=scoring, n_jobs=-1)
search.fit(X_train_fe, y_train)

sign = 1 if TASK == "classification" else -1
print(f"Best parameters: {search.best_params_}")
print(
    f"Best CV {scoring}: {sign * search.best_score_:.4f}  (selection-biased - confirm with cross_val_score if it matters)"
)

# Use the tuned pipeline from here on
final_pipeline = search.best_estimator_
final_model = final_pipeline.named_steps["model"]
final_core = final_model.estimator if isinstance(final_model, CalibratedClassifierCV) else final_model
assert isinstance(final_core, ALLOWED_MODELS[EDITION])

# 7. Train on All Data and Write the Submission

The final model is fitted on the **whole** training set, then applied to the test set. The submission file must have exactly the columns and the row order of `sample_submission.csv`.

The cell also writes an **MD5 checksum** of the file: after "Restart Kernel and Run All" the checksum must be identical — that is the reproducibility check performed when notebooks are verified.

In [ ]:
final_pipeline.fit(X_train_fe, y_train)

if TASK == "classification":
    predictions = final_pipeline.predict_proba(X_test_fe)[:, 1]  # probabilities of class 1 - never predict()!
else:
    predictions = final_pipeline.predict(X_test_fe)

submission = pd.DataFrame({ID_COLUMN: test[ID_COLUMN], PREDICTION_COLUMN: predictions})

# Format checks against sample_submission
assert list(submission.columns) == list(sample_submission.columns[:2])
assert len(submission) == len(sample_submission)
assert submission[ID_COLUMN].tolist() == sample_submission[ID_COLUMN].tolist(), "row order must match sample_submission"
assert submission[PREDICTION_COLUMN].notna().all(), "predictions contain NaN"
if TASK == "classification":
    assert submission[PREDICTION_COLUMN].between(0, 1).all(), "probabilities must lie in [0, 1]"

OUTPUT_DIR.mkdir(exist_ok=True)
safe_team = "".join(ch if ch.isalnum() else "_" for ch in TEAM_NAME)
submission_path = OUTPUT_DIR / f"submission_edition{EDITION}_{TASK}_{safe_team}.csv"
submission.to_csv(submission_path, index=False, float_format="%.6f")

checksum = hashlib.md5(submission_path.read_bytes()).hexdigest()
print(f"Submission written to: {submission_path}")
print(f"Rows: {len(submission)}, MD5: {checksum}")
print("\nFirst rows:")
print(submission.head())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(predictions, bins=40, edgecolor="black", alpha=0.75)
ax.set_title("Distribution of the Submitted Predictions", fontsize=13, fontweight="bold")
ax.set_xlabel(PREDICTION_COLUMN)
plt.tight_layout()
plt.show()

# 8. Pre-Upload Checklist

- [ ] Header cell filled in: team name, edition, dataset, public score of the **selected** submission
- [ ] `EDITION` and `TASK` in the configuration cell match the header
- [ ] The notebook loads the raw Kaggle files only — no external data, no hand-labelled test rows
- [ ] All predictions come from a model of the edition's family (the assertions passed); other models were used at most for auxiliary tasks (e.g. feature selection)
- [ ] `Kernel → Restart Kernel and Run All Cells` completes without errors in the course environment
- [ ] The MD5 checksum after the clean run equals the checksum of the file you selected on Kaggle
- [ ] The notebook is your team's own work (public tutorials as a source of ideas are fine; copied solutions are not)
- [ ] Uploaded before the edition closes (Sunday 23:59, Warsaw time) — one notebook per dataset